In [1]:
import pandas as pd

path_CTA_L_Rides = r"C:\Users\danhm\Desktop\MetroFlow\CTA_Riderships_Daily_L.csv"

CTA_L_RIDES_DF = pd.read_csv(
    path_CTA_L_Rides,
    usecols=["station_id", "stationname", "date", "daytype", "rides"],
    dtype={
        "station_id": "Int64",
        "stationname": "category",
        "daytype": "category",
        "rides": "string",  # <- cargar como texto para no fallar con "1,059"
    },
    na_values=["", "TRAM2"],
)

# parse de fecha (rápido y controlado)
CTA_L_RIDES_DF["date"] = pd.to_datetime(CTA_L_RIDES_DF["date"], format="%m/%d/%Y", errors="coerce")

# limpieza óptima (vectorizada) y conversión segura
CTA_L_RIDES_DF["rides"] = (
    CTA_L_RIDES_DF["rides"]
      .str.replace(",", "", regex=False)  # "1,059" -> "1059"
      .str.strip()
)
CTA_L_RIDES_DF["rides"] = pd.to_numeric(CTA_L_RIDES_DF["rides"], errors="coerce").astype("Int64")
CTA_L_RIDES_DF["day"] = (
    pd.to_datetime(CTA_L_RIDES_DF["date"], format="%m/%d/%Y", errors="coerce")
      .dt.day_name()
)

CTA_L_RIDES_DF.head()


,station_id,stationname,date,daytype,rides,day
0,40350,UIC-Halsted,2001-01-01,U,273,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,Monday
2,40760,Granville,2001-01-01,U,1059,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,Monday
4,40090,Damen-Brown,2001-01-01,U,411,Monday


In [2]:
pip install geopandas leafmap lonboard -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

path_CTA_L_Stops = r"C:\Users\danhm\Desktop\MetroFlow\CTA_List_of_L_Stops.csv"

cols = [
    "STOP_ID","DIRECTION_ID","STOP_NAME","STATION_NAME","STATION_DESCRIPTIVE_NAME",
    "MAP_ID","ADA","RED","BLUE","G","BRN","P","Y","Pnk","O","Location"
]

stops = pd.read_csv(
    path_CTA_L_Stops,
    usecols=cols,
    dtype="string",                       # <- evita que truene por tipos raros
    na_values=["", "NA", "N/A", "NULL"],
)

# 1) Llaves numéricas (MAP_ID = FK para rides)
stops["STOP_ID"] = pd.to_numeric(stops["STOP_ID"], errors="coerce").astype("Int64")
stops["MAP_ID"]  = pd.to_numeric(stops["MAP_ID"],  errors="coerce").astype("Int64")

# 2) Booleans (true/false -> boolean)
bool_cols = ["ADA","RED","BLUE","G","BRN","P","Y","Pnk","O"]
for c in bool_cols:
    stops[c] = stops[c].str.strip().str.lower().map({"true": True, "false": False}).astype("boolean")

# 3) Location -> lat/lon (formato: "(41.857908, -87.669147)" = (lat, lon))
stops[["lat", "lon"]] = stops["Location"].str.extract(r"\(\s*([-\d.]+)\s*,\s*([-\d.]+)\s*\)")
stops["lat"] = pd.to_numeric(stops["lat"], errors="coerce")
stops["lon"] = pd.to_numeric(stops["lon"], errors="coerce")

# 4) Limpieza ligera de texto + categorías
text_cols = ["STOP_NAME","STATION_NAME","STATION_DESCRIPTIVE_NAME","DIRECTION_ID"]
for c in text_cols:
    stops[c] = stops[c].str.strip()

stops["DIRECTION_ID"] = stops["DIRECTION_ID"].astype("category")
stops["STATION_NAME"] = stops["STATION_NAME"].astype("category")

CTA_L_STOPS_DF = stops
CTA_L_STOPS_DF.head()

,STOP_ID,DIRECTION_ID,STOP_NAME,STATION_NAME,STATION_DESCRIPTIVE_NAME,MAP_ID,ADA,RED,BLUE,G,BRN,P,Y,Pnk,O,Location,lat,lon
0,30162,W,18th (54th/Cermak-bound),18th,18th (Pink Line),40830,True,False,False,False,False,False,False,True,False,"(41.857908, -87.669147)",41.857908,-87.669147
1,30161,E,18th (Loop-bound),18th,18th (Pink Line),40830,True,False,False,False,False,False,False,True,False,"(41.857908, -87.669147)",41.857908,-87.669147
2,30022,N,35th/Archer (Loop-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,False,False,False,False,True,"(41.829353, -87.680622)",41.829353,-87.680622
3,30023,S,35th/Archer (Midway-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,False,False,False,False,True,"(41.829353, -87.680622)",41.829353,-87.680622
4,30213,N,35-Bronzeville-IIT (Harlem-bound),35th-Bronzeville-IIT,35th-Bronzeville-IIT (Green Line),41120,True,False,False,True,False,False,False,False,False,"(41.831677, -87.625826)",41.831677,-87.625826


In [4]:
CTA_RIDES_JOIN_STATIONS_DF = CTA_L_RIDES_DF.merge(
    CTA_L_STOPS_DF,
    left_on="station_id",
    right_on="MAP_ID",
    how="left"
)

In [5]:
CTA_RIDES_JOIN_STATIONS_DF.head(20)

,station_id,stationname,date,daytype,rides,day,STOP_ID,DIRECTION_ID,STOP_NAME,STATION_NAME,...,BLUE,G,BRN,P,Y,Pnk,O,Location,lat,lon
0,40350,UIC-Halsted,2001-01-01,U,273,Monday,30068,E,UIC-Halsted (O'Hare-bound),UIC-Halsted,...,True,False,False,False,False,False,False,"(41.875474, -87.649707)",41.875474,-87.649707
1,40350,UIC-Halsted,2001-01-01,U,273,Monday,30069,W,UIC-Halsted (Forest Pk-bound),UIC-Halsted,...,True,False,False,False,False,False,False,"(41.875474, -87.649707)",41.875474,-87.649707
2,41130,Halsted-Orange,2001-01-01,U,306,Monday,30216,S,Halsted (Midway-bound),Halsted,...,False,False,False,False,False,False,True,"(41.84678, -87.648088)",41.84678,-87.648088
3,41130,Halsted-Orange,2001-01-01,U,306,Monday,30215,N,Halsted (Loop-bound),Halsted,...,False,False,False,False,False,False,True,"(41.84678, -87.648088)",41.84678,-87.648088
4,40760,Granville,2001-01-01,U,1059,Monday,30148,S,Granville (95th-bound),Granville,...,False,False,False,False,False,False,False,"(41.993664, -87.659202)",41.993664,-87.659202
5,40760,Granville,2001-01-01,U,1059,Monday,30147,N,Granville (Howard-bound),Granville,...,False,False,False,False,False,False,False,"(41.993664, -87.659202)",41.993664,-87.659202
6,40070,Jackson/Dearborn,2001-01-01,U,649,Monday,30014,N,Jackson/Dearborn (O'Hare-bound),Jackson,...,True,False,False,False,False,False,False,"(41.878183, -87.629296)",41.878183,-87.629296
7,40070,Jackson/Dearborn,2001-01-01,U,649,Monday,30015,S,Jackson/Dearborn (Forest Pk-bound),Jackson,...,True,False,False,False,False,False,False,"(41.878183, -87.629296)",41.878183,-87.629296
8,40090,Damen-Brown,2001-01-01,U,411,Monday,30019,S,Damen (Loop-bound),Damen,...,False,False,True,False,False,False,False,"(41.966286, -87.678639)",41.966286,-87.678639
9,40090,Damen-Brown,2001-01-01,U,411,Monday,30018,N,Damen (Kimball-bound),Damen,...,False,False,True,False,False,False,False,"(41.966286, -87.678639)",41.966286,-87.678639


In [6]:
CTA_RIDES_JOIN_STATIONS_DF_2023 = CTA_RIDES_JOIN_STATIONS_DF[CTA_RIDES_JOIN_STATIONS_DF['date'].dt.year == 2023]
CTA_RIDES_JOIN_STATIONS_DF_2023.to_csv('CTA_RIDES_JOIN_STATIONS_2023.csv')

In [7]:
CTA_RIDES_JOIN_STATIONS_DF.size

65002968

In [8]:
import pandas as pd

# --- paths ---
OD_PATH = "C:\\Users\\danhm\\Desktop\\MetroFlow\\il_od_main_JT00_2023.csv\\il_od_main_JT00_2023.csv"   # <-- cambia al nombre real de tu archivo OD
XWALK_URL = "https://lehd.ces.census.gov/data/lodes/LODES8/il/il_xwalk.csv.gz"

CHICAGO_STPLC = "1714000"  # Chicago city: state 17 + place 14000

# --- read OD ---
od = pd.read_csv(
    OD_PATH,
    dtype={"w_geocode": "string", "h_geocode": "string"},
)

# --- read xwalk (solo columnas necesarias para ahorrar RAM) ---
xw = pd.read_csv(
    XWALK_URL,
    compression="gzip",
    dtype={"tabblk2020": "string", "stplc": "string"},
    usecols=["tabblk2020", "stplc", "blklatdd", "blklondd"],
)

# --- filtrar a bloques dentro de Chicago ---
xw_chi = xw.loc[xw["stplc"] == CHICAGO_STPLC, ["tabblk2020", "blklatdd", "blklondd"]]

# --- merge para WORK block ---
od = od.merge(
    xw_chi.rename(columns={"tabblk2020": "w_geocode", "blklatdd": "w_lat", "blklondd": "w_lon"}),
    on="w_geocode",
    how="left",
)

# --- merge para HOME block ---
od = od.merge(
    xw_chi.rename(columns={"tabblk2020": "h_geocode", "blklatdd": "h_lat", "blklondd": "h_lon"}),
    on="h_geocode",
    how="left",
)

# Opciones de filtrado:
# A) Solo viajes completamente dentro de Chicago (home Y work en Chicago)
od_both_in_chicago = od.dropna(subset=["w_lat", "h_lat"])

# B) Viajes que tocan Chicago (home O work en Chicago): commutes entrando/saliendo
od_either_in_chicago = od[od["w_lat"].notna() | od["h_lat"].notna()]

# Guardar
od_both_in_chicago.to_csv("lodes_od_chicago_both.csv", index=False)
od_either_in_chicago.to_csv("lodes_od_chicago_either.csv", index=False)

# Tabla única de bloques Chicago -> lat/lon (útil para luego mapear a estaciones CTA)
blocks = pd.concat([
    od_both_in_chicago[["w_geocode","w_lat","w_lon"]].rename(columns={"w_geocode":"geocode","w_lat":"lat","w_lon":"lon"}),
    od_both_in_chicago[["h_geocode","h_lat","h_lon"]].rename(columns={"h_geocode":"geocode","h_lat":"lat","h_lon":"lon"}),
]).drop_duplicates("geocode")

blocks.to_csv("chicago_blocks_lonlat.csv", index=False)
print("Listo:", len(od_both_in_chicago), "filas (both) |", len(blocks), "bloques únicos")


Listo: 659934 filas (both) | 35308 bloques únicos


In [9]:
Origin_Destination_Jobs_Chicago = pd.read_csv(
    "C:\\Users\\danhm\\Desktop\\MetroFlow\\lodes_od_chicago_both.csv",
    usecols=["S000","SA01","SA02","SA03","SE01","SE02","SE03","SI01","SI02","SI03","createdate","w_lat","w_lon","h_lat","h_lon"],
    dtype="string",                       # <- evita que truene por tipos raros
    na_values=["", "NA", "N/A", "NULL"],
)

Origin_Destination_Jobs_Chicago["w_lat"] = pd.to_numeric(Origin_Destination_Jobs_Chicago["w_lat"], errors="coerce")
Origin_Destination_Jobs_Chicago["w_lon"] = pd.to_numeric(Origin_Destination_Jobs_Chicago["w_lon"], errors="coerce")

Origin_Destination_Jobs_Chicago["h_lat"] = pd.to_numeric(Origin_Destination_Jobs_Chicago["h_lat"], errors="coerce")
Origin_Destination_Jobs_Chicago["h_lon"] = pd.to_numeric(Origin_Destination_Jobs_Chicago["h_lon"], errors="coerce")

In [10]:
Origin_Destination_Jobs_Chicago.head(20)

,S000,SA01,SA02,SA03,SE01,SE02,SE03,SI01,SI02,SI03,createdate,w_lat,w_lon,h_lat,h_lon
0,1,0,0,1,0,1,0,0,1,0,20251202,42.022659,-87.666748,41.986426,-87.656352
1,1,0,1,0,0,1,0,0,1,0,20251202,42.022659,-87.666748,41.943965,-87.815211
2,1,0,1,0,0,1,0,0,1,0,20251202,42.022659,-87.666748,41.797258,-87.600889
3,1,0,1,0,0,1,0,0,1,0,20251202,42.022659,-87.666748,41.786286,-87.62104
4,1,0,1,0,0,1,0,0,0,1,20251202,42.022759,-87.674666,42.021475,-87.668372
5,1,0,1,0,0,0,1,0,0,1,20251202,42.022759,-87.674666,42.022759,-87.674666
6,1,0,0,1,0,0,1,0,0,1,20251202,42.022759,-87.674666,42.021854,-87.672775
7,1,0,0,1,0,1,0,0,0,1,20251202,42.022759,-87.674666,41.970207,-87.694771
8,1,1,0,0,0,1,0,0,0,1,20251202,42.022759,-87.674666,41.930901,-87.640055
9,1,0,1,0,0,0,1,0,0,1,20251202,42.022759,-87.674666,41.971379,-87.777358


In [11]:
import geopandas
import matplotlib.pyplot as plt
from lonboard import Map, ScatterplotLayer
import lonboard as lb
import leafmap

In [12]:
import numpy as np
import geopandas as gpd
from lonboard import Map, ScatterplotLayer
import ipywidgets as widgets
from IPython.display import display

df = Origin_Destination_Jobs_Chicago.copy()

# --- Peso del flujo: usa S000 si existe; si no, cuenta filas ---
weight = "S000" if "S000" in df.columns else None
if weight is None:
    df["_jobs"] = 1
    weight = "_jobs"

# --- Limpieza ---
df = df.dropna(subset=["w_lat","w_lon","h_lat","h_lon", weight]).copy()
df[weight] = df[weight].astype(float)

# --- Agregar: total empleos por punto ---
work = (df.groupby(["w_lat","w_lon"], as_index=False)[weight]
          .sum()
          .rename(columns={"w_lat":"lat","w_lon":"lon", weight:"jobs"}))

home = (df.groupby(["h_lat","h_lon"], as_index=False)[weight]
          .sum()
          .rename(columns={"h_lat":"lat","h_lon":"lon", weight:"jobs"}))

# --- Radios: más pequeños + robustos (cap al p95) ---
def make_radius(jobs_series):
    q95 = float(jobs_series.quantile(0.95)) if len(jobs_series) else 1.0
    q95 = q95 if q95 > 0 else 1.0
    # 12m → ~110m (más discreto)
    return (12 + 100 * np.sqrt(np.minimum(jobs_series, q95) / q95)).to_numpy(dtype=np.float32)

work_radius = make_radius(work["jobs"])
home_radius = make_radius(home["jobs"])

# (Opcional) limitar puntos por performance
MAX_POINTS = 7000
if len(work) > MAX_POINTS:
    work = work.assign(_r=work_radius).sort_values("jobs", ascending=False).head(MAX_POINTS)
    work_radius = work["_r"].to_numpy(dtype=np.float32)
    work = work.drop(columns=["_r"])
if len(home) > MAX_POINTS:
    home = home.assign(_r=home_radius).sort_values("jobs", ascending=False).head(MAX_POINTS)
    home_radius = home["_r"].to_numpy(dtype=np.float32)
    home = home.drop(columns=["_r"])

# --- GeoDataFrames con SOLO el campo que quieres en el tooltip ---
gdf_work = gpd.GeoDataFrame(
    {"Jobs #": work["jobs"].round().astype(int).to_numpy()},
    geometry=gpd.points_from_xy(work["lon"], work["lat"]),
    crs="EPSG:4326",
)

gdf_home = gpd.GeoDataFrame(
    {"Homes #": home["jobs"].round().astype(int).to_numpy()},
    geometry=gpd.points_from_xy(home["lon"], home["lat"]),
    crs="EPSG:4326",
)

# --- Capas (menos opacas + pequeñas) ---
work_layer = ScatterplotLayer.from_geopandas(
    gdf_work,
    get_fill_color=[255, 149, 0, 105],   # naranja suave (work)
    get_line_color=[0, 0, 0, 35],
    get_line_width=1,
    get_radius=work_radius,
    radius_min_pixels=1,
    radius_max_pixels=16,
    opacity=0.55,
    pickable=True,
    auto_highlight=True,
)

home_layer = ScatterplotLayer.from_geopandas(
    gdf_home,
    get_fill_color=[0, 122, 255, 105],   # azul suave (home)
    get_line_color=[0, 0, 0, 35],
    get_line_width=1,
    get_radius=home_radius,
    radius_min_pixels=1,
    radius_max_pixels=16,
    opacity=0.55,
    pickable=True,
    auto_highlight=True,
)

# --- Basemap sin Basemap import ---
POSITRON = "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"

m = Map(
    [work_layer, home_layer],
    basemap_style=POSITRON,
    height=820,
    show_tooltip=True,
    show_side_panel=False,   # si solo quieres la ventanita, apaga el panel lateral
    picking_radius=6,
    view_state={"longitude": -87.6298, "latitude": 41.8781, "zoom": 10.6},
)

# --- Controles show/hide ---
chk_work = widgets.Checkbox(value=True, description="Work (w_) — Jobs")
chk_home = widgets.Checkbox(value=True, description="Home (h_) — Homes")

def toggle_layers(_=None):
    work_layer.visible = chk_work.value
    home_layer.visible = chk_home.value

chk_work.observe(toggle_layers, names="value")
chk_home.observe(toggle_layers, names="value")

display(widgets.HBox([chk_work, chk_home]))
m


In [13]:
# ============================
# CTA + Heatmap continuo animado (1 sola celda)
# - Frío = ausencia (no se pinta)
# - Calor = # personas (densidad)
# Requiere:
#   - Archivo: CTA_List_of_L_Stops.csv
#   - DataFrame: Origin_Destination_Jobs_Chicago con h_lat,h_lon,w_lat,w_lon (+ opcional S000)
# ============================

import pandas as pd
import numpy as np
import re
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point, LineString

from lonboard import Map, ScatterplotLayer, PathLayer, HeatmapLayer
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# 0) Config
# ============================================================
LINE_COLS = {
    "Red": "RED",
    "Blue": "BLUE",
    "Green": "G",
    "Brown": "BRN",
    "Purple": "P",
    "Yellow": "Y",
    "Pink": "Pnk",
    "Orange": "O",
}

RGBA = {
    "Red":    [220,  30,  38, 180],
    "Blue":   [  0, 122, 255, 180],
    "Green":  [ 52, 199,  89, 180],
    "Brown":  [150,  75,   0, 180],
    "Purple": [175,  82, 222, 180],
    "Yellow": [255, 204,   0, 180],
    "Pink":   [255,  45,  85, 180],
    "Orange": [255, 149,   0, 180],
}

POSITRON = "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"

# Loop edge (para cerrar circuito)
LOOP_U, LOOP_V = 40730, 40040
LOOP_LINES = {"Orange", "Pink", "Purple", "Brown"}

# Modelo: TODOS usan CTA
K_NEIGHBORS = 3
SCALE_ASSIGN_M = 700.0

# Heatmap look (ajustable en sliders)
HEAT_RADIUS_PX = 60
HEAT_INTENSITY = 2.0
HEAT_THRESHOLD = 0.01
EDGE_SAMPLES = 7  # puntos por arista para continuidad visual

# Horas: 1am-11am y 1pm-11pm
HOURS_AM = list(range(1, 12))
HOURS_PM = list(range(13, 24))
HOURS = HOURS_AM + HOURS_PM


# ============================================================
# 1) Estaciones: CSV -> station-level
# ============================================================
def _parse_location_to_latlon(val):
    m = re.search(r"\(\s*([-\d\.]+)\s*,\s*([-\d\.]+)\s*\)", str(val))
    if not m:
        return (np.nan, np.nan)
    return (float(m.group(1)), float(m.group(2)))  # (lat, lon)

def prepare_station_level_df(stops_df: pd.DataFrame) -> pd.DataFrame:
    df = stops_df.copy()

    latlon = df["Location"].apply(_parse_location_to_latlon)
    df["LATITUDE"] = latlon.apply(lambda x: x[0])
    df["LONGITUDE"] = latlon.apply(lambda x: x[1])

    for col in LINE_COLS.values():
        if df[col].dtype == object:
            df[col] = (
                df[col].astype(str).str.strip().str.lower()
                .map({"true": True, "false": False})
            )
        df[col] = df[col].fillna(False).astype(bool)

    agg = {
        "STATION_NAME": "first",
        "STATION_DESCRIPTIVE_NAME": "first",
        "LATITUDE": "first",
        "LONGITUDE": "first",
    }
    for col in LINE_COLS.values():
        agg[col] = "max"

    stations = (
        df.groupby("MAP_ID", as_index=False)
          .agg(agg)
          .dropna(subset=["LATITUDE", "LONGITUDE"])
    )
    stations["MAP_ID"] = stations["MAP_ID"].astype(int)
    return stations


# ============================================================
# 2) Haversine (metros) + matriz vectorizada
# ============================================================
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    phi1 = np.radians(lat1); phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlmb/2.0)**2
    return float(2.0 * R * np.arcsin(np.sqrt(a)))

def haversine_matrix_m(points_lon, points_lat, stations_lon, stations_lat):
    R = 6371000.0
    p_lat = np.radians(points_lat).reshape(-1, 1)
    p_lon = np.radians(points_lon).reshape(-1, 1)
    s_lat = np.radians(stations_lat).reshape(1, -1)
    s_lon = np.radians(stations_lon).reshape(1, -1)
    dlat = s_lat - p_lat
    dlon = s_lon - p_lon
    a = np.sin(dlat/2.0)**2 + np.cos(p_lat) * np.cos(s_lat) * np.sin(dlon/2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(a))


# ============================================================
# 3) Grafo por línea (MST) + edge especial loop
# ============================================================
def build_line_mst_graph(stations_df: pd.DataFrame, line_name: str) -> nx.Graph:
    sub = stations_df[stations_df[LINE_COLS[line_name]]].copy()

    G = nx.Graph(line=line_name)

    for _, r in sub.iterrows():
        nid = int(r["MAP_ID"])
        lon = float(r["LONGITUDE"])
        lat = float(r["LATITUDE"])
        G.add_node(
            nid,
            pos=(lon, lat),
            station_name=r["STATION_NAME"],
            station_desc=r["STATION_DESCRIPTIVE_NAME"],
        )

    nodes = list(G.nodes(data=True))
    for i in range(len(nodes)):
        u, du = nodes[i]
        lon1, lat1 = du["pos"]
        for j in range(i + 1, len(nodes)):
            v, dv = nodes[j]
            lon2, lat2 = dv["pos"]
            G.add_edge(u, v, weight=haversine_m(lon1, lat1, lon2, lat2))

    T = nx.minimum_spanning_tree(G, weight="weight")
    T.graph["line"] = line_name
    return T

def add_special_connection(G: nx.Graph, u: int, v: int, tag="loop"):
    if (u not in G.nodes) or (v not in G.nodes):
        return False
    lon1, lat1 = G.nodes[u]["pos"]
    lon2, lat2 = G.nodes[v]["pos"]
    w = haversine_m(lon1, lat1, lon2, lat2)
    G.add_edge(u, v, weight=w, kind=tag)
    return True


# ============================================================
# 4) Grafo -> layers lonboard
# ============================================================
def graph_to_lonboard_layers(T: nx.Graph, rgba):
    pos = nx.get_node_attributes(T, "pos")

    node_rows = []
    for n, data in T.nodes(data=True):
        lon, lat = pos[n]
        node_rows.append({
            "MAP_ID": n,
            "Line": T.graph.get("line"),
            "Station": data.get("station_name"),
            "Desc": data.get("station_desc"),
            "geometry": Point(lon, lat),
        })
    nodes_gdf = gpd.GeoDataFrame(node_rows, crs="EPSG:4326")

    edge_rows = []
    for u, v, data in T.edges(data=True):
        lon1, lat1 = pos[u]
        lon2, lat2 = pos[v]
        edge_rows.append({
            "Line": T.graph.get("line"),
            "u": u, "v": v,
            "weight_m": float(data.get("weight", 0.0)),
            "kind": data.get("kind", "mst"),
            "geometry": LineString([(lon1, lat1), (lon2, lat2)]),
        })
    edges_gdf = gpd.GeoDataFrame(edge_rows, crs="EPSG:4326")

    stations_layer = ScatterplotLayer.from_geopandas(
        nodes_gdf[["MAP_ID","Line","Station","Desc","geometry"]],
        get_fill_color=rgba,
        get_line_color=[0, 0, 0, 40],
        get_line_width=1,
        radius_units="pixels",
        get_radius=4,
        radius_min_pixels=2,
        radius_max_pixels=9,
        opacity=0.85,
        pickable=True,
        auto_highlight=True,
    )

    edges_layer = PathLayer.from_geopandas(
        edges_gdf[["Line","u","v","weight_m","kind","geometry"]],
        get_color=rgba,
        width_units="pixels",
        get_width=2,
        width_min_pixels=1,
        width_max_pixels=6,
        opacity=0.55,
        pickable=False,
    )

    return stations_layer, edges_layer


# ============================================================
# 5) Punto -> K estaciones con prob exp(-d/scale)
# ============================================================
def k_nearest_station_probs(unique_points_lon, unique_points_lat,
                            stations_lon, stations_lat, station_ids,
                            k=3, scale_m=700.0):
    D = haversine_matrix_m(unique_points_lon, unique_points_lat, stations_lon, stations_lat)
    k = min(k, D.shape[1])

    idx = np.argpartition(D, kth=k-1, axis=1)[:, :k]
    dsel = np.take_along_axis(D, idx, axis=1)

    order = np.argsort(dsel, axis=1)
    idx = np.take_along_axis(idx, order, axis=1)
    dsel = np.take_along_axis(dsel, order, axis=1)

    w = np.exp(-dsel / float(scale_m))
    w_sum = w.sum(axis=1, keepdims=True)
    w = w / np.where(w_sum == 0, 1.0, w_sum)

    nn_ids = station_ids[idx]
    return nn_ids.astype(np.int32), w.astype(np.float32)


# ============================================================
# 6) OD -> matriz flujo estación-estación (sin strings)
# ============================================================
def compute_station_flow_matrix_all_use_cta(
    od_df: pd.DataFrame,
    stations_df: pd.DataFrame,
    k=3,
    scale_assign_m=700.0,
    weight_col=None
):
    od = od_df.copy()

    if weight_col is None or weight_col not in od.columns:
        od["_w"] = 1.0
        weight_col = "_w"

    od = od.dropna(subset=["h_lat","h_lon","w_lat","w_lon", weight_col]).copy()
    od[weight_col] = od[weight_col].astype(float)

    h_lat = od["h_lat"].astype(float).round(6).to_numpy()
    h_lon = od["h_lon"].astype(float).round(6).to_numpy()
    w_lat = od["w_lat"].astype(float).round(6).to_numpy()
    w_lon = od["w_lon"].astype(float).round(6).to_numpy()
    trips = od[weight_col].to_numpy(dtype=np.float32)

    station_ids = stations_df["MAP_ID"].astype(int).to_numpy()
    s_lon = stations_df["LONGITUDE"].astype(float).to_numpy()
    s_lat = stations_df["LATITUDE"].astype(float).to_numpy()
    S = len(station_ids)

    # unique homes/works por filas
    home_xy = np.c_[h_lon, h_lat]
    uniq_home_xy, inv_home = np.unique(home_xy, axis=0, return_inverse=True)
    home_u_lon, home_u_lat = uniq_home_xy[:, 0], uniq_home_xy[:, 1]

    work_xy = np.c_[w_lon, w_lat]
    uniq_work_xy, inv_work = np.unique(work_xy, axis=0, return_inverse=True)
    work_u_lon, work_u_lat = uniq_work_xy[:, 0], uniq_work_xy[:, 1]

    home_ids_u, home_w_u = k_nearest_station_probs(home_u_lon, home_u_lat, s_lon, s_lat, station_ids, k=k, scale_m=scale_assign_m)
    work_ids_u, work_w_u = k_nearest_station_probs(work_u_lon, work_u_lat, s_lon, s_lat, station_ids, k=k, scale_m=scale_assign_m)

    U_ids = home_ids_u[inv_home]  # (N,k)
    U_w   = home_w_u[inv_home]    # (N,k)
    V_ids = work_ids_u[inv_work]  # (N,k)
    V_w   = work_w_u[inv_work]    # (N,k)

    # MAP_ID -> índice con sorting+searchsorted
    sort_idx = np.argsort(station_ids)
    sids_sorted = station_ids[sort_idx]

    def map_ids_to_index(ids_flat):
        pos = np.searchsorted(sids_sorted, ids_flat)
        pos = np.clip(pos, 0, len(sids_sorted)-1)
        ok = (sids_sorted[pos] == ids_flat)
        out = np.full(ids_flat.shape, -1, dtype=np.int32)
        out[ok] = sort_idx[pos[ok]]
        return out

    u_flat = np.repeat(U_ids[:, :, None], k, axis=2).reshape(-1).astype(np.int32)
    v_flat = np.repeat(V_ids[:, None, :], k, axis=1).reshape(-1).astype(np.int32)
    val_flat = (trips[:, None, None] * U_w[:, :, None] * V_w[:, None, :]).reshape(-1).astype(np.float32)

    u_idx = map_ids_to_index(u_flat)
    v_idx = map_ids_to_index(v_flat)

    mask = (u_idx >= 0) & (v_idx >= 0)
    flow = np.zeros((S, S), dtype=np.float32)
    np.add.at(flow, (u_idx[mask], v_idx[mask]), val_flat[mask])

    return flow, station_ids


# ============================================================
# 7) Flow station->station -> carga por arista (shortest paths)
# ============================================================
def build_edge_loads_from_station_flows(flow_matrix: np.ndarray, station_ids: np.ndarray, G_net: nx.Graph):
    paths = dict(nx.all_pairs_dijkstra_path(G_net, weight="weight"))
    edge_load = {}

    nz = np.transpose(np.nonzero(flow_matrix))
    for i, j in nz:
        if i == j:
            continue
        f = float(flow_matrix[i, j])
        if f <= 0:
            continue
        src = int(station_ids[i])
        dst = int(station_ids[j])
        try:
            p = paths[src][dst]
        except Exception:
            continue

        for a, b in zip(p[:-1], p[1:]):
            key = (a, b) if a < b else (b, a)
            edge_load[key] = edge_load.get(key, 0.0) + f

    return edge_load


# ============================================================
# 8) Edge loads -> puntos para heatmap continuo
# ============================================================
def edge_loads_to_sample_points(edge_load: dict, G_net: nx.Graph, samples_per_edge=7):
    rows = []
    ts = np.linspace(0.10, 0.90, samples_per_edge)

    for (u, v), load in edge_load.items():
        if u not in G_net.nodes or v not in G_net.nodes:
            continue
        lon1, lat1 = G_net.nodes[u]["pos"]
        lon2, lat2 = G_net.nodes[v]["pos"]

        load_each = float(load) / float(samples_per_edge)
        for t in ts:
            lon = (1 - t) * lon1 + t * lon2
            lat = (1 - t) * lat1 + t * lat2
            rows.append({"base_people": load_each, "geometry": Point(lon, lat)})

    return gpd.GeoDataFrame(rows, crs="EPSG:4326")


# ============================================================
# 9) Perfil horario (shares)
# ============================================================
def _gauss(x, mu, sig):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * ((x - mu) / sig) ** 2)

def make_hour_shares(hours_am, hours_pm):
    # AM pico ~ 8am
    am = np.array(hours_am, dtype=float)
    am_raw = _gauss(am, 8.0, 1.6)
    am_raw[am < 5] *= 0.08
    am_raw = am_raw / am_raw.sum()

    # PM pico ~ 5pm
    pm = np.array(hours_pm, dtype=float)
    pm_raw = _gauss(pm, 17.0, 1.8)
    pm_raw[pm < 15] *= 0.35
    pm_raw = pm_raw / pm_raw.sum()

    shares = {}
    for h, s in zip(hours_am, am_raw): shares[int(h)] = float(s)
    for h, s in zip(hours_pm, pm_raw): shares[int(h)] = float(s)
    return shares

def hour_label(h):
    if h == 0: return "12am"
    if h < 12: return f"{h}am"
    if h == 12: return "12pm"
    return f"{h-12}pm"


# ============================================================
# 10) RUN
# ============================================================
if "Origin_Destination_Jobs_Chicago" not in globals():
    raise NameError("No encuentro 'Origin_Destination_Jobs_Chicago' en memoria. Cárgalo antes de correr esta celda.")

# estaciones
stops = pd.read_csv("CTA_List_of_L_Stops.csv")
stations = prepare_station_level_df(stops)

# líneas CTA
cta_graphs = {}
cta_layers = {}
cta_layer_list = []

for line_name in LINE_COLS.keys():
    T = build_line_mst_graph(stations, line_name)
    if line_name in LOOP_LINES:
        add_special_connection(T, LOOP_U, LOOP_V, tag="loop")

    cta_graphs[line_name] = T
    st_layer, ed_layer = graph_to_lonboard_layers(T, RGBA[line_name])
    cta_layers[line_name] = {"stations": st_layer, "edges": ed_layer}
    cta_layer_list += [ed_layer, st_layer]

# grafo combinado para rutas
G_net = nx.Graph()
for g in cta_graphs.values():
    G_net = nx.compose(G_net, g)
add_special_connection(G_net, LOOP_U, LOOP_V, tag="loop")

# matriz station->station (TODOS usan CTA)
od = Origin_Destination_Jobs_Chicago.copy()
weight_col = "S000" if "S000" in od.columns else None
flow_matrix, station_ids = compute_station_flow_matrix_all_use_cta(
    od, stations, k=K_NEIGHBORS, scale_assign_m=SCALE_ASSIGN_M, weight_col=weight_col
)

# edge loads + puntos heatmap
edge_load = build_edge_loads_from_station_flows(flow_matrix, station_ids, G_net)
base_pts = edge_loads_to_sample_points(edge_load, G_net, samples_per_edge=EDGE_SAMPLES)
if len(base_pts) == 0:
    raise RuntimeError("No se generaron puntos para heatmap. Revisa conectividad de G_net o tus IDs.")

base_weights = base_pts["base_people"].to_numpy(dtype=np.float32)  # (M,)

# shares por hora (solo escalamos pesos, NO duplicamos geometría)
shares = make_hour_shares(HOURS_AM, HOURS_PM)

# Heatmap: solo geometría, el peso se lo damos como array (lonboard-style)
heat_geom = base_pts[["geometry"]].copy()
h0 = HOURS[0]
w0 = base_weights * np.float32(shares.get(int(h0), 0.0))

heat_layer = HeatmapLayer.from_geopandas(
    heat_geom,
    get_weight=w0,              # <- ARRAY, NO string
    radius_pixels=HEAT_RADIUS_PX,
    intensity=HEAT_INTENSITY,
    threshold=HEAT_THRESHOLD,
    opacity=0.95,
    pickable=False,
)

# mapa (heat debajo)
m = Map(
    [heat_layer] + cta_layer_list,
    basemap_style=POSITRON,
    height=820,
    show_tooltip=True,
    show_side_panel=False,
    picking_radius=6,
    view_state={"longitude": -87.6298, "latitude": 41.8781, "zoom": 10.6},
)

# ============================================================
# 11) UI
# ============================================================
chk_heat = widgets.Checkbox(value=True, description="Heatmap (People)")
chk_all = widgets.Checkbox(value=True, description="CTA Lines (All)")
chk_edges = widgets.Checkbox(value=True, description="CTA Edges")
chk_stations = widgets.Checkbox(value=True, description="CTA Stations")
line_checks = {ln: widgets.Checkbox(value=True, description=ln) for ln in LINE_COLS.keys()}

play = widgets.Play(value=0, min=0, max=len(HOURS)-1, step=1, interval=450)
slider = widgets.IntSlider(value=0, min=0, max=len(HOURS)-1, step=1, description="t", continuous_update=False)
widgets.jslink((play, "value"), (slider, "value"))

lbl = widgets.HTML(value=f"<b>Hour: {hour_label(HOURS[0])}</b>")

rad = widgets.IntSlider(value=HEAT_RADIUS_PX, min=10, max=160, step=1, description="Radius(px)")
inten = widgets.FloatSlider(value=HEAT_INTENSITY, min=0.1, max=8.0, step=0.1, description="Intensity")
thres = widgets.FloatSlider(value=HEAT_THRESHOLD, min=0.0, max=0.2, step=0.005, description="Threshold")

def apply_visibility():
    heat_layer.visible = chk_heat.value
    for ln, chk in line_checks.items():
        cta_layers[ln]["edges"].visible = chk.value and chk_edges.value
        cta_layers[ln]["stations"].visible = chk.value and chk_stations.value

def toggle_all(_=None):
    v = chk_all.value
    for ln in line_checks:
        line_checks[ln].value = v
    apply_visibility()

def toggle_any(_=None):
    chk_all.value = all(chk.value for chk in line_checks.values())
    apply_visibility()

def on_time_change(change):
    i = int(change["new"])
    h = int(HOURS[i])
    share = np.float32(shares.get(h, 0.0))
    heat_layer.get_weight = (base_weights * share).astype(np.float32)  # <- update dinámico
    lbl.value = f"<b>Hour: {hour_label(h)}</b>"

def on_heat_params(_):
    heat_layer.radius_pixels = float(rad.value)
    heat_layer.intensity = float(inten.value)
    heat_layer.threshold = float(thres.value)

chk_heat.observe(toggle_any, names="value")
chk_all.observe(toggle_all, names="value")
chk_edges.observe(toggle_any, names="value")
chk_stations.observe(toggle_any, names="value")
for chk in line_checks.values():
    chk.observe(toggle_any, names="value")

slider.observe(on_time_change, names="value")
for w in [rad, inten, thres]:
    w.observe(on_heat_params, names="value")

apply_visibility()

ui = widgets.VBox([
    widgets.HBox([chk_heat, chk_all, chk_edges, chk_stations]),
    widgets.HBox([
        widgets.VBox(list(line_checks.values())[:4]),
        widgets.VBox(list(line_checks.values())[4:]),
    ]),
    widgets.HBox([play, slider, lbl]),
    widgets.HBox([rad, inten, thres]),
])

display(ui)
m


In [14]:
# ============================================================
# CTA "FLOWING" HEATMAP (15-min frames) — 1 cell
# - Heat moves along the metro network using travel time on edges.
# - AM: home -> work (1:00–11:00 every 15min)
# - PM: work -> home (13:00–23:00 every 15min)
# ============================================================

import pandas as pd
import numpy as np
import re
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point, LineString

from lonboard import Map, ScatterplotLayer, PathLayer, HeatmapLayer
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# CONFIG (ajusta aquí)
# ============================================================
POSITRON = "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"

LINE_COLS = {
    "Red": "RED",
    "Blue": "BLUE",
    "Green": "G",
    "Brown": "BRN",
    "Purple": "P",
    "Yellow": "Y",
    "Pink": "Pnk",
    "Orange": "O",
}
RGBA = {
    "Red":    [220,  30,  38, 180],
    "Blue":   [  0, 122, 255, 180],
    "Green":  [ 52, 199,  89, 180],
    "Brown":  [150,  75,   0, 180],
    "Purple": [175,  82, 222, 180],
    "Yellow": [255, 204,   0, 180],
    "Pink":   [255,  45,  85, 180],
    "Orange": [255, 149,   0, 180],
}

# Loop edge (cierra circuito)
LOOP_U, LOOP_V = 40730, 40040
LOOP_LINES = {"Orange", "Pink", "Purple", "Brown"}

# Asignación home/work -> estación (hard nearest para que el flujo sea nítido)
ROUND_COORDS = 6
ASSIGN_CHUNK = 8000  # chunking para no reventar memoria

# Movimiento en la red (para que “fluya”)
SPEED_MPS = 8.0      # ~28.8 km/h
DWELL_S = 25.0       # dwell por estación (aprox)
MIN_PAIR_FLOW = 10.0 # ignora pares con flujo muy chico (performance)
TOP_PAIRS = 12000    # None para usar todos (si tu dataset es enorme, deja 12000 o baja)

# Heatmap
EDGE_SAMPLES = 10
HEAT_RADIUS_PX = 80
HEAT_INTENSITY = 2.8
HEAT_THRESHOLD = 0.006

# Timeline
DT_MIN = 15
DT_S = DT_MIN * 60
ALL_BINS = np.arange(0, 24*60, DT_MIN)  # 0..(24h-15min) in minutes, len=96
N_BINS = len(ALL_BINS)

# Frames visibles (lo que pediste)
AM_MIN = np.arange(60, 11*60 + 1, DT_MIN)    # 1:00..11:00
PM_MIN = np.arange(13*60, 23*60 + 1, DT_MIN) # 13:00..23:00
FRAME_MINUTES = np.concatenate([AM_MIN, PM_MIN])
FRAME_BINS = (FRAME_MINUTES // DT_MIN).astype(int)

# Perfil depart (para que haya más gente en ciertos tiempos)
AM_PEAK_MIN = 8*60     # 8:00
PM_PEAK_MIN = 17*60    # 17:00
AM_SIGMA_MIN = 60
PM_SIGMA_MIN = 70


# ============================================================
# CHEQUEOS
# ============================================================
if "Origin_Destination_Jobs_Chicago" not in globals():
    raise NameError("No encuentro 'Origin_Destination_Jobs_Chicago' en memoria. Cárgalo antes de correr esta celda.")


# ============================================================
# 1) Estaciones (CTA CSV)
# ============================================================
def _parse_location_to_latlon(val):
    m = re.search(r"\(\s*([-\d\.]+)\s*,\s*([-\d\.]+)\s*\)", str(val))
    if not m:
        return (np.nan, np.nan)
    return (float(m.group(1)), float(m.group(2)))  # (lat, lon)

def prepare_station_level_df(stops_df: pd.DataFrame) -> pd.DataFrame:
    df = stops_df.copy()
    latlon = df["Location"].apply(_parse_location_to_latlon)
    df["LATITUDE"] = latlon.apply(lambda x: x[0])
    df["LONGITUDE"] = latlon.apply(lambda x: x[1])

    for col in LINE_COLS.values():
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip().str.lower().map({"true": True, "false": False})
        df[col] = df[col].fillna(False).astype(bool)

    agg = {
        "STATION_NAME": "first",
        "STATION_DESCRIPTIVE_NAME": "first",
        "LATITUDE": "first",
        "LONGITUDE": "first",
    }
    for col in LINE_COLS.values():
        agg[col] = "max"

    stations = (
        df.groupby("MAP_ID", as_index=False)
          .agg(agg)
          .dropna(subset=["LATITUDE", "LONGITUDE"])
    )
    stations["MAP_ID"] = stations["MAP_ID"].astype(int)
    return stations

stops = pd.read_csv("CTA_List_of_L_Stops.csv")
stations = prepare_station_level_df(stops)

station_ids = stations["MAP_ID"].astype(int).to_numpy()
s_lon = stations["LONGITUDE"].astype(float).to_numpy()
s_lat = stations["LATITUDE"].astype(float).to_numpy()


# ============================================================
# 2) Haversine (vectorizado por chunk)
# ============================================================
def haversine_matrix_m(points_lon, points_lat, stations_lon, stations_lat):
    R = 6371000.0
    p_lat = np.radians(points_lat).reshape(-1, 1)
    p_lon = np.radians(points_lon).reshape(-1, 1)
    s_lat = np.radians(stations_lat).reshape(1, -1)
    s_lon = np.radians(stations_lon).reshape(1, -1)
    dlat = s_lat - p_lat
    dlon = s_lon - p_lon
    a = np.sin(dlat/2.0)**2 + np.cos(p_lat) * np.cos(s_lat) * np.sin(dlon/2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(a))

def assign_unique_points_nearest_station(u_lon, u_lat, s_lon, s_lat, s_ids, chunk=8000):
    out = np.empty(len(u_lon), dtype=np.int32)
    for i in range(0, len(u_lon), chunk):
        j = min(i + chunk, len(u_lon))
        D = haversine_matrix_m(u_lon[i:j], u_lat[i:j], s_lon, s_lat)
        idx = np.argmin(D, axis=1)
        out[i:j] = s_ids[idx]
    return out


# ============================================================
# 3) Grafo CTA por línea (MST) + loop
# ============================================================
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    phi1 = np.radians(lat1); phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlmb/2.0)**2
    return float(2.0 * R * np.arcsin(np.sqrt(a)))

def build_line_mst_graph(stations_df: pd.DataFrame, line_name: str) -> nx.Graph:
    sub = stations_df[stations_df[LINE_COLS[line_name]]].copy()
    G = nx.Graph(line=line_name)

    for _, r in sub.iterrows():
        nid = int(r["MAP_ID"])
        lon = float(r["LONGITUDE"])
        lat = float(r["LATITUDE"])
        G.add_node(nid, pos=(lon, lat),
                   station_name=r["STATION_NAME"],
                   station_desc=r["STATION_DESCRIPTIVE_NAME"])

    nodes = list(G.nodes(data=True))
    for i in range(len(nodes)):
        u, du = nodes[i]
        lon1, lat1 = du["pos"]
        for j in range(i+1, len(nodes)):
            v, dv = nodes[j]
            lon2, lat2 = dv["pos"]
            G.add_edge(u, v, weight=haversine_m(lon1, lat1, lon2, lat2))

    T = nx.minimum_spanning_tree(G, weight="weight")
    T.graph["line"] = line_name
    return T

def add_special_connection(G: nx.Graph, u: int, v: int, tag="loop"):
    if u not in G.nodes or v not in G.nodes:
        return False
    lon1, lat1 = G.nodes[u]["pos"]
    lon2, lat2 = G.nodes[v]["pos"]
    w = haversine_m(lon1, lat1, lon2, lat2)
    G.add_edge(u, v, weight=w, kind=tag)
    return True

def graph_to_lonboard_layers(T: nx.Graph, rgba):
    pos = nx.get_node_attributes(T, "pos")

    nodes = []
    for n, data in T.nodes(data=True):
        lon, lat = pos[n]
        nodes.append({
            "MAP_ID": n,
            "Line": T.graph.get("line"),
            "Station": data.get("station_name"),
            "Desc": data.get("station_desc"),
            "geometry": Point(lon, lat),
        })
    nodes_gdf = gpd.GeoDataFrame(nodes, crs="EPSG:4326")

    edges = []
    for u, v, data in T.edges(data=True):
        lon1, lat1 = pos[u]
        lon2, lat2 = pos[v]
        edges.append({
            "Line": T.graph.get("line"),
            "u": u, "v": v,
            "kind": data.get("kind","mst"),
            "weight_m": float(data.get("weight",0.0)),
            "geometry": LineString([(lon1, lat1),(lon2, lat2)])
        })
    edges_gdf = gpd.GeoDataFrame(edges, crs="EPSG:4326")

    st_layer = ScatterplotLayer.from_geopandas(
        nodes_gdf[["MAP_ID","Line","Station","Desc","geometry"]],
        get_fill_color=rgba,
        get_line_color=[0,0,0,35],
        get_line_width=1,
        radius_units="pixels",
        get_radius=4,
        radius_min_pixels=2,
        radius_max_pixels=9,
        opacity=0.85,
        pickable=True,
        auto_highlight=True,
    )
    ed_layer = PathLayer.from_geopandas(
        edges_gdf[["Line","u","v","kind","weight_m","geometry"]],
        get_color=rgba,
        width_units="pixels",
        get_width=2,
        width_min_pixels=1,
        width_max_pixels=6,
        opacity=0.40,
        pickable=False,
    )
    return st_layer, ed_layer

cta_graphs = {}
cta_layers = {}
cta_layer_list = []

for ln in LINE_COLS.keys():
    T = build_line_mst_graph(stations, ln)
    if ln in LOOP_LINES:
        add_special_connection(T, LOOP_U, LOOP_V, tag="loop")
    cta_graphs[ln] = T
    st_layer, ed_layer = graph_to_lonboard_layers(T, RGBA[ln])
    cta_layers[ln] = {"stations": st_layer, "edges": ed_layer}
    cta_layer_list += [ed_layer, st_layer]

# combinado
G_net = nx.Graph()
for g in cta_graphs.values():
    G_net = nx.compose(G_net, g)
add_special_connection(G_net, LOOP_U, LOOP_V, tag="loop")


# ============================================================
# 4) OD -> station pairs (hard nearest) + aggregate
# ============================================================
od = Origin_Destination_Jobs_Chicago.copy()
weight_col = "S000" if "S000" in od.columns else None
if weight_col is None:
    od["_w"] = 1.0
    weight_col = "_w"

od = od.dropna(subset=["h_lat","h_lon","w_lat","w_lon", weight_col]).copy()
od[weight_col] = od[weight_col].astype(float)

h_lon = od["h_lon"].astype(float).round(ROUND_COORDS).to_numpy()
h_lat = od["h_lat"].astype(float).round(ROUND_COORDS).to_numpy()
w_lon = od["w_lon"].astype(float).round(ROUND_COORDS).to_numpy()
w_lat = od["w_lat"].astype(float).round(ROUND_COORDS).to_numpy()
trip_w = od[weight_col].to_numpy(dtype=np.float32)

# unique homes/works (numérico)
home_xy = np.c_[h_lon, h_lat]
uniq_home_xy, inv_home = np.unique(home_xy, axis=0, return_inverse=True)
work_xy = np.c_[w_lon, w_lat]
uniq_work_xy, inv_work = np.unique(work_xy, axis=0, return_inverse=True)

home_sid_u = assign_unique_points_nearest_station(
    uniq_home_xy[:,0], uniq_home_xy[:,1], s_lon, s_lat, station_ids, chunk=ASSIGN_CHUNK
)
work_sid_u = assign_unique_points_nearest_station(
    uniq_work_xy[:,0], uniq_work_xy[:,1], s_lon, s_lat, station_ids, chunk=ASSIGN_CHUNK
)

home_sid = home_sid_u[inv_home].astype(np.int32)
work_sid = work_sid_u[inv_work].astype(np.int32)

pairs_df = pd.DataFrame({"u": home_sid, "v": work_sid, "w": trip_w})
pairs_df = pairs_df[pairs_df["u"] != pairs_df["v"]]
pairs_agg = pairs_df.groupby(["u","v"], as_index=False)["w"].sum()
pairs_agg = pairs_agg[pairs_agg["w"] >= MIN_PAIR_FLOW].copy()

if TOP_PAIRS is not None and len(pairs_agg) > TOP_PAIRS:
    pairs_agg = pairs_agg.sort_values("w", ascending=False).head(TOP_PAIRS).copy()

pairs_agg.reset_index(drop=True, inplace=True)


# ============================================================
# 5) Departure profiles (15-min bins)
# ============================================================
def gaussian_on_bins(bin_minutes, mu_min, sigma_min):
    x = bin_minutes.astype(float)
    y = np.exp(-0.5 * ((x - mu_min) / sigma_min) ** 2)
    y = y / max(y.sum(), 1e-12)
    return y.astype(np.float32)

am_prob = np.zeros(N_BINS, dtype=np.float32)
pm_prob = np.zeros(N_BINS, dtype=np.float32)

am_bins = (AM_MIN // DT_MIN).astype(int)
pm_bins = (PM_MIN // DT_MIN).astype(int)

am_prob_vals = gaussian_on_bins(AM_MIN, AM_PEAK_MIN, AM_SIGMA_MIN)
pm_prob_vals = gaussian_on_bins(PM_MIN, PM_PEAK_MIN, PM_SIGMA_MIN)

am_prob[am_bins] = am_prob_vals
pm_prob[pm_bins] = pm_prob_vals

# normaliza (por si queda algo raro)
am_prob = am_prob / max(am_prob.sum(), 1e-12)
pm_prob = pm_prob / max(pm_prob.sum(), 1e-12)


# ============================================================
# 6) Edge indexing + shortest paths
# ============================================================
edges_list = []
edge_to_idx = {}
for a, b in G_net.edges():
    e = (a, b) if a < b else (b, a)
    if e not in edge_to_idx:
        edge_to_idx[e] = len(edges_list)
        edges_list.append(e)

E = len(edges_list)

# precompute all-pairs shortest paths (nodos ~ estaciones)
paths = dict(nx.all_pairs_dijkstra_path(G_net, weight="weight"))

def edge_time_s(dist_m):
    return float(dist_m) / float(SPEED_MPS) + float(DWELL_S)

# cache: (u,v) -> (forward_shifts, reverse_shifts)
# each shifts list: [(edge_idx, shift0, frac), ...]
shift_cache = {}

def get_shifts_for_pair(u, v):
    key = (int(u), int(v))
    if key in shift_cache:
        return shift_cache[key]

    try:
        p = paths[int(u)][int(v)]
    except Exception:
        shift_cache[key] = ([], [])
        return shift_cache[key]

    # forward: compute mid offsets
    cum = 0.0
    mids = []   # (edge_idx, mid_s)
    for a, b in zip(p[:-1], p[1:]):
        e = (a, b) if a < b else (b, a)
        if e not in edge_to_idx:
            continue
        dist = float(G_net[a][b].get("weight", 0.0))
        t = edge_time_s(dist)
        mid = cum + 0.5 * t
        cum += t
        mids.append((edge_to_idx[e], mid))

    total_t = cum
    if not mids:
        shift_cache[key] = ([], [])
        return shift_cache[key]

    # convert mid_s -> (shift0, frac) in bins of DT_S, with linear interpolation
    fwd = []
    rev = []
    for ei, mid_s in mids:
        x = mid_s / DT_S
        s0 = int(np.floor(x))
        frac = float(x - s0)

        fwd.append((ei, s0, frac))

        # reverse mid is total_t - mid_s
        xr = (total_t - mid_s) / DT_S
        r0 = int(np.floor(xr))
        rfrac = float(xr - r0)
        rev.append((ei, r0, rfrac))

    shift_cache[key] = (fwd, rev)
    return shift_cache[key]


# ============================================================
# 7) Build edge_load_bins[edge, bin] (AM + PM)
# ============================================================
edge_load_bins = np.zeros((E, N_BINS), dtype=np.float32)

def add_flow_to_edge_bins(shifts, flow_val, depart_prob):
    # shifts: list[(ei, shift0, frac)]
    # depart_prob: (N_BINS,)
    if flow_val <= 0:
        return
    for (ei, s0, frac) in shifts:
        if s0 >= N_BINS:
            continue
        # main slice
        n = N_BINS - s0
        edge_load_bins[ei, s0:] += np.float32(flow_val * (1.0 - frac)) * depart_prob[:n]
        # next bin slice (interp)
        if frac > 0 and (s0 + 1) < N_BINS:
            n2 = N_BINS - (s0 + 1)
            edge_load_bins[ei, s0+1:] += np.float32(flow_val * frac) * depart_prob[:n2]

# iterate station pairs
for row in pairs_agg.itertuples(index=False):
    u = int(row.u); v = int(row.v); f = float(row.w)
    fwd, rev = get_shifts_for_pair(u, v)
    if fwd:
        # AM: home->work
        add_flow_to_edge_bins(fwd, f, am_prob)
        # PM: work->home (reverse order in time)
        add_flow_to_edge_bins(rev, f, pm_prob)

# ============================================================
# 8) Sample points along edges -> heatmap points + weights per bin
# ============================================================
pt_rows = []
pt_edge_idx = []

ts = np.linspace(0.10, 0.90, EDGE_SAMPLES)
for ei, (u, v) in enumerate(edges_list):
    if u not in G_net.nodes or v not in G_net.nodes:
        continue
    lon1, lat1 = G_net.nodes[u]["pos"]
    lon2, lat2 = G_net.nodes[v]["pos"]
    for t in ts:
        lon = (1-t)*lon1 + t*lon2
        lat = (1-t)*lat1 + t*lat2
        pt_rows.append({"geometry": Point(lon, lat)})
        pt_edge_idx.append(ei)

pts_gdf = gpd.GeoDataFrame(pt_rows, crs="EPSG:4326")
pt_edge_idx = np.asarray(pt_edge_idx, dtype=np.int32)

def point_weights_for_bin(bin_idx):
    e_w = edge_load_bins[:, int(bin_idx)].astype(np.float32) / np.float32(EDGE_SAMPLES)
    return e_w[pt_edge_idx]

# init frame
bin0 = int(FRAME_BINS[0])
w0 = point_weights_for_bin(bin0)

heat_layer = HeatmapLayer.from_geopandas(
    pts_gdf[["geometry"]],
    get_weight=w0,  # array float
    radius_pixels=HEAT_RADIUS_PX,
    intensity=HEAT_INTENSITY,
    threshold=HEAT_THRESHOLD,
    opacity=0.95,
    pickable=False,
)

# ============================================================
# 9) MAP + UI
# ============================================================
m = Map(
    [heat_layer] + cta_layer_list,
    basemap_style=POSITRON,
    height=820,
    show_tooltip=True,
    show_side_panel=False,
    picking_radius=6,
    view_state={"longitude": -87.6298, "latitude": 41.8781, "zoom": 10.6},
)

def label_time_from_bin(b):
    minutes = int(b) * DT_MIN
    hh = minutes // 60
    mm = minutes % 60
    ampm = "am" if hh < 12 else "pm"
    hh12 = hh % 12
    if hh12 == 0:
        hh12 = 12
    return f"{hh12}:{mm:02d}{ampm}"

# toggles CTA
chk_heat = widgets.Checkbox(value=True, description="Heatmap (flowing)")
chk_all = widgets.Checkbox(value=True, description="CTA Lines (All)")
chk_edges = widgets.Checkbox(value=True, description="CTA Edges")
chk_stations = widgets.Checkbox(value=True, description="CTA Stations")
line_checks = {ln: widgets.Checkbox(value=True, description=ln) for ln in LINE_COLS.keys()}

# animation (15-min)
play = widgets.Play(value=0, min=0, max=len(FRAME_BINS)-1, step=1, interval=220)
slider = widgets.IntSlider(value=0, min=0, max=len(FRAME_BINS)-1, step=1,
                           description="frame", continuous_update=False)
widgets.jslink((play, "value"), (slider, "value"))

lbl = widgets.HTML(value=f"<b>Time: {label_time_from_bin(bin0)}</b>")

# heat tuning
rad = widgets.IntSlider(value=HEAT_RADIUS_PX, min=10, max=220, step=1, description="Radius(px)")
inten = widgets.FloatSlider(value=HEAT_INTENSITY, min=0.1, max=12.0, step=0.1, description="Intensity")
thres = widgets.FloatSlider(value=HEAT_THRESHOLD, min=0.0, max=0.2, step=0.002, description="Threshold")

def apply_visibility():
    heat_layer.visible = chk_heat.value
    for ln, chk in line_checks.items():
        cta_layers[ln]["edges"].visible = chk.value and chk_edges.value
        cta_layers[ln]["stations"].visible = chk.value and chk_stations.value

def toggle_all(_=None):
    v = chk_all.value
    for ln in line_checks:
        line_checks[ln].value = v
    apply_visibility()

def toggle_any(_=None):
    chk_all.value = all(chk.value for chk in line_checks.values())
    apply_visibility()

def on_frame(change):
    i = int(change["new"])
    b = int(FRAME_BINS[i])
    heat_layer.get_weight = point_weights_for_bin(b)
    lbl.value = f"<b>Time: {label_time_from_bin(b)}</b>"

def on_heat_params(_):
    heat_layer.radius_pixels = float(rad.value)
    heat_layer.intensity = float(inten.value)
    heat_layer.threshold = float(thres.value)

# observers
chk_heat.observe(toggle_any, names="value")
chk_all.observe(toggle_all, names="value")
chk_edges.observe(toggle_any, names="value")
chk_stations.observe(toggle_any, names="value")
for chk in line_checks.values():
    chk.observe(toggle_any, names="value")

slider.observe(on_frame, names="value")
for w in [rad, inten, thres]:
    w.observe(on_heat_params, names="value")

apply_visibility()

ui = widgets.VBox([
    widgets.HBox([chk_heat, chk_all, chk_edges, chk_stations]),
    widgets.HBox([
        widgets.VBox(list(line_checks.values())[:4]),
        widgets.VBox(list(line_checks.values())[4:]),
    ]),
    widgets.HBox([play, slider, lbl]),
    widgets.HBox([rad, inten, thres]),
])

display(ui)
m
